In [52]:
from utils.database_utils import generate_database_and_retriever
from neo4j import GraphDatabase
import json
from utils.rag import parse_documents

URI = "bolt://localhost:7687"
AUTH = ("neo4j", "123456789")


def get_graph_context(graph_driver, retrieved_nodes):
    """
    Takes the output from your MultiVectorRetriever and
    fetches a 2-hop neighborhood for each node.
    """

    all_knowledge_blocks = []
    with graph_driver.session() as session:
        connections = {}
        entities = set({})

        for full_node_params in retrieved_nodes:
            params = {
                "name": full_node_params["name"],
                "label": full_node_params["type"],
                "description": full_node_params["description"],
            }
            # Assuming retrieved_nodes are Document objects from your retriever
            # We pull the name or ID from the metadata

            query = """
            MATCH (source)
            WHERE source.name = $name 
            AND $label IN labels(source) 
            AND source.description = $description

            MATCH path = (source)-[*1..2]-(neighbor)
            WHERE source <> neighbor
            AND NONE(lbl IN labels(neighbor) WHERE lbl IN ['text', 'image', 'table', 'document'])
            AND NONE(rel IN relationships(path) WHERE type(rel) = 'BELONGS_TO')

            WITH source, neighbor, collect(path) AS paths
            UNWIND paths as p
            UNWIND relationships(p) as rel
            RETURN DISTINCT collect({
                subject: startNode(rel).name,
                subject_description: startNode(rel).description, 
                predicate: type(rel),
                object: endNode(rel).name,
                object_description: endNode(rel).description
            }) AS connection_paths
            """

            def format_rel(rel_type):
                return rel_type.replace("_", " ").lower()

            results = session.run(query, **params)

            # Collect the results:

            for record in results:
                for connection in record["connection_paths"]:
                    subject = connection["subject"]
                    subject_desc = connection["subject_description"]
                    predicate = connection["predicate"]
                    object_ = connection["object"]
                    object_desc = connection["object_description"]

                    # Adding entities
                    entities.add((subject, subject_desc))
                    entities.add((object_, object_desc))

                    # Adding connections
                    if (subject, subject_desc) not in connections:
                        connections[(subject, subject_desc)] = {}

                    if predicate not in connections[(subject, subject_desc)]:
                        connections[(subject, subject_desc)][predicate] = set([])

                    connections[(subject, subject_desc)][predicate].add(
                        (object_, object_desc)
                    )

        entities_narrative = []
        for entity in entities:
            entities_narrative.append(
                f"\t'{entity[0]}': {entity[1] or 'No description.'}"
            )

        connections_narrative = []
        for subject, subject_desc in connections.keys():
            for predicate, object_object_desc in connections[
                (subject, subject_desc)
            ].items():
                for object_, object_desc in object_object_desc:
                    connections_narrative.append(
                        f"\t'{subject}' {format_rel(predicate)} '{object_}'"
                    )
            # Create a dense, summarized block
        summary_block = (
            f"ENTITIES:\n{',\n'.join(entities_narrative)}:\n",
            f"RELATIONSHIPS:\n{',\n'.join(connections_narrative)}\n",
        )
        all_knowledge_blocks.append("\n".join(summary_block))

    return "\n".join(all_knowledge_blocks)


def parse_nodes(nodes_retrieved):
    nodes_parsed = []
    for node in nodes_retrieved:
        nodes_parsed.append(json.loads(node.decode("utf-8")))
    return nodes_parsed


from langchain_ollama import OllamaLLM
from langchain_core.messages import SystemMessage, HumanMessage


def summarize_graph_context(nodes_context, model_name="gemma3:latest"):
    system_prompt = """

        You are an expert in knowledge synthesis and technical summarization.

        Your task is to transform structured knowledge (entities + relationships) into a dense, high-signal summary optimized for retrieval in a RAG system.

        ---

        ### OBJECTIVE:
        Generate a compact, information-rich representation that:
        - Preserves all critical technical facts and metrics
        - Consolidates duplicate or conflicting values (prefer most consistent or repeated signals)
        - Removes redundancy and noise
        - Filters out invalid, weak, or illogical relationships
        - Clearly separates model structure, inputs/outputs, and performance

        ---

        ### INPUT:
        You will receive:
        1. ENTITIES: Named concepts with descriptions
        2. RELATIONSHIPS: Triplets connecting entities

        ---

        ### PROCESSING RULES:
        - Deduplicate metrics (e.g., multiple F1/accuracy values → summarize as ranges or most representative values)
        - Ignore contradictory or after reasoning low-confidence relationships unless strongly supported
        - Normalize synonyms (e.g., "optimal shade", "optimal threshold", "cutoff value" → one concept)

        ---

        ### OUTPUT FORMAT:

        1. **Dense Summary (5–8 sentences max)**  
        - Highly compressed, technical narrative  
        - Must include: model types, inputs, outputs, key metrics, thresholds, and interpretability approach  

        2. **Structured Key Insights (bullet points)**  
        - Max 8 bullets  
        - Each bullet = one atomic, high-value insight  
        - Prefer normalized terminology and grouped metrics  

        ---

        ### STYLE:
        - Technical and precise
        - High signal-to-noise ratio
        - No repetition
        - No explanations or commentary
        - No hallucinated connections

        ---

        ### CONSTRAINT:
        Only use information grounded in the provided entities and relationships.
        Do not infer beyond the data unless necessary for normalization.
        Return only the final summary and bullet points.

    """
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"{nodes_context}"),
    ]
    model = OllamaLLM(model=model_name)
    return model.invoke(messages)


def retrieve_context_for_nodes(nodes_retrieved):
    nodes_parsed = parse_nodes(nodes_retrieved)
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        nodes_context = get_graph_context(driver, nodes_parsed)

    nodes_context = summarize_graph_context(nodes_context)
    return nodes_context


def parse_communities(communities_retrieved):
    all_communities = []
    for com in communities_retrieved:
        all_communities.append(json.loads(com.decode("utf-8")))

    summaries = []
    for com in all_communities:
        summaries.append(com.get("summary", ""))

    return "\n----\n".join(summaries)


# Retrievers

You have to initiate 4 different retrievers: 
 - For regular rag
 - For nodes related to the search query
 - For mid level communities
 - For global level communities

In [ ]:
regular_rag_retriever = generate_database_and_retriever(main_folder="./localdb")

nodes_retriever = generate_database_and_retriever(main_folder="./localdb/node_db")

mid_level_retriever = generate_database_and_retriever(
    main_folder="./localdb/mid_communities"
)

global_level_retriever = generate_database_and_retriever(
    main_folder="./localdb/global_communities"
)

In [75]:
user_query = (
    "Explain me the relationship between linear model and model interpretability."
)

In [76]:
docs_retrieved = regular_rag_retriever.invoke(user_query)
regular_rag_documents = parse_documents(docs_retrieved)

mid_level_communities_retrieved = mid_level_retriever.invoke(user_query)
summaries_mid_level_communities = parse_communities(mid_level_communities_retrieved)

global_level_communities_retrieved = global_level_retriever.invoke(user_query)
summaries_global_level_communities = parse_communities(
    global_level_communities_retrieved
)

nodes_retrieved = nodes_retriever.invoke(user_query)
nodes_context = retrieve_context_for_nodes(nodes_retrieved)

In [77]:
RAG_SYSTEM_PROMPT = """
You are an expert analytical assistant. Your task is to provide a comprehensive and accurate answer to the user's query based strictly on the provided context. 
INSTRUCTIONS:
1. Synthesize the answer by starting with the specific facts (<Specific_Documents> and <Visual_Context>).
2. Use the <Entity_Nodes> to clarify any relationships between actors or concepts.
3. Weave in the <Mid_Level_Context> and <Global_Context> to provide background, explain the "why", or fill in gaps if the specific documents are sparse.
4. If the different contexts contradict each other, prioritize <Specific_Documents> for exact facts, but note the discrepancy in your answer.
5. Do not include information that is not present in the contexts.
"""

RAG_USER_PROMPT = """


    The context is derived from multiple sources: high-level summaries, specific entity data, raw document snippets, and visual inputs.

    <Global_Context>
    [Use this to understand the overarching environment or dataset as a whole]
    {summaries_global_level_communities}
    </Global_Context>

    <Mid_Level_Context>
    [Use this to understand thematic clusters, regional trends, or sub-topics relevant to the query]
    {summaries_mid_level_communities}
    </Mid_Level_Context>

    <Entity_Nodes>
    [Use this to define key actors, terms, or direct relationships mentioned in the query]
    {nodes_context}
    </Entity_Nodes>

    <Specific_Documents>
    [Use this for exact quotes, granular facts, and highly specific details]
    {regular_rag_documents_texts}
    </Specific_Documents>

    <Visual_Context>
    [If images are attached, incorporate their contents into your reasoning]
    (Images attached in multimodal payload)
    </Visual_Context>

    ---
    USER QUERY: 
    {user_query}
    ---

"""


def build_prompt(
    input_dict, system_prompt=RAG_SYSTEM_PROMPT, user_prompt=RAG_USER_PROMPT
):
    """
    Constructs the multimodal prompt array.
    Expects input_dict to contain: 'user_query', 'regular_rag_documents',
    'nodes_context', 'summaries_mid_level_communities', 'summaries_global_level_communities'
    """

    # Safely extract GraphRAG context
    global_context = input_dict.get(
        "summaries_global_level_communities", "No global context retrieved."
    )
    mid_context = input_dict.get(
        "summaries_mid_level_communities", "No mid-level context retrieved."
    )
    nodes_context = input_dict.get("nodes_context", "No entity context retrieved.")

    # Format regular RAG texts
    regular_texts_list = input_dict.get("regular_rag_documents", {}).get("texts", [])
    formatted_regular_texts = "\n\n".join(
        [f"[Source {i + 1}]: {t}" for i, t in enumerate(regular_texts_list)]
    )

    # Inject all context into the XML template
    text_content = user_prompt.format(
        summaries_global_level_communities=global_context,
        summaries_mid_level_communities=mid_context,
        nodes_context=nodes_context,
        regular_rag_documents_texts=formatted_regular_texts,
        user_query=input_dict.get("user_query", ""),
    )

    # Construct the multimodal content list for the HumanMessage
    user_content = [
        {
            "type": "text",
            "text": text_content,
        }
    ]

    # Append images as base64 strings
    images_list = input_dict.get("regular_rag_documents", {}).get("images", [])
    for img_b64 in images_list:
        user_content.append(
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"},
            }
        )

    return [SystemMessage(content=system_prompt), HumanMessage(content=user_content)]


In [78]:
messages = build_prompt(
    {
        "user_query": user_query,
        "regular_rag_documents": regular_rag_documents,
        "nodes_context": nodes_context,
        "summaries_mid_level_communities": summaries_mid_level_communities,
        "summaries_global_level_communities": summaries_global_level_communities,
    }
)
model = OllamaLLM(model="gemma3:12b")
result = model.invoke(messages)

In [79]:
print(result)

The relationship between linear regression and model interpretability revolves around using a linear regression model as a "proxy model" to understand a more complex "original model." The goal is to achieve "original model interpretability" by creating simpler, interpretable models that approximate the behavior of the original.

Here's a breakdown of the relationship, based on the provided context:

**1. Proxy Model Approach:** The community focuses on techniques to improve the interpretability of a complex original model through proxy models. A linear regression model serves as one such proxy.

**2. Linear Regression as a Proxy:** A "proxy regression linear model" is specifically derived and described by equation 5.1.  This model acts as a simplified substitute for the original, more complicated model.  A proxy decision tree model also contributes to achieving original model interpretability. Figure 5.22 visually represents this proxy decision tree model.

**3. Performance Metrics & E